In [1]:
import json
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import os

def load_result():
    possible_paths = [
        "result.json",
        "outputs/result.json"
    ]
    for p in possible_paths:
        if os.path.exists(p):
            with open(p, encoding='utf-8') as f:
                return json.load(f)
    raise Exception("no finding result.json")

data = load_result()
summary = data.get("summary", {})
parsed_trades = data.get("parsed_trades", [])
compliance_results = data.get("compliance_results", [])
novel_trade_ids = summary.get("novel_instrument_trade_ids", [])

In [2]:
# Chart 1: Portfolio compliance heatmap
import plotly.express as px
import json

# --------------- 1. pull result.json ---------------
with open("result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

parsed_trades = data.get("parsed_trades", [])
compliance_results = data.get("compliance_results", [])

compliance_map = {}
for item in compliance_results:
    tid = item["trade_id"]
    regime = item["regime"]
    status = item["status"]

    if tid not in compliance_map:
        compliance_map[tid] = {}
    compliance_map[tid][regime] = status

# --------------- 2. prepare data ---------------
trade_ids = [t["trade_id"] for t in parsed_trades]
regimes = ["CFTC", "EMIR"]

status_list = ["COMPLIANT", "CONDITIONAL", "NOT_APPLICABLE", "NONCOMPLIANT"]
color_list = ["#22c55e", "#eab308", "#94a3b8", "#ef4444"]
status_to_num = {s: i for i, s in enumerate(status_list)}

z_data = []
text_data = []
for tid in trade_ids:
    row_num = []
    row_text = []
    # CFTC
    cftc = compliance_map.get(tid, {}).get("CFTC", "NONCOMPLIANT")
    row_num.append(status_to_num[cftc])
    row_text.append(cftc)
    # EMIR
    emir = compliance_map.get(tid, {}).get("EMIR", "NONCOMPLIANT")
    row_num.append(status_to_num[emir])
    row_text.append(emir)

    z_data.append(row_num)
    text_data.append(row_text)

# --------------- 3. draw heatmap ---------------
print('1. Portfolio compliance heatmap')
fig = px.imshow(
    z_data,
    x=regimes,
    y=trade_ids,
    color_continuous_scale=color_list,
    range_color=[0, 3],
    title="Portfolio Compliance Heatmap",
    height=800,
    labels=dict(color="Status"),
    aspect="auto"  # 让单元格比例更合理
)

# --------------- 4. correct text ---------------
fig.update_traces(
    text=text_data,
    texttemplate="%{text}",
    textfont={"size": 10, "color": "white"},
    hovertemplate="Trade: %{y}<br>Regime: %{x}<br>Status: %{text}<extra></extra>"
)

# --------------- 5. correct colour---------------
fig.update_layout(
    coloraxis_colorbar=dict(
        tickvals=[0, 1, 2, 3],
        ticktext=status_list,
        title="Status",
        title_side="right"
    ),
    plot_bgcolor="#121212",
    paper_bgcolor="#121212",
    font_color="white",
    xaxis_title="Regime",
    yaxis_title="Trade ID",
    title_x=0.5
)

fig.show()

1. Portfolio compliance heatmap


Comparing the compliance status of 28 trades under the CFTC (U.S.) and EMIR (EU) regimes, most trades are marked "NONCOMPLIANT". Only T017 and T021 are "COMPLIANT" under CFTC; T026 and T028 are "CONDITIONAL". T027 is "NOT_APPLICABLE" under both, highlighting cross-jurisdictional compliance differences and widespread non-compliance risks.

In [3]:
# Chart 2: Error frequency chart
print("2. Error frequency chart")

errors = []
for c in compliance_results:
    for f in c.get("findings", []):
        errors.append(f.get("field", "unknown"))

if errors:
    df_err = pd.Series(errors).value_counts().reset_index()
    df_err.columns = ["field", "count"]
    fig2 = px.bar(df_err, x="count", y="field", orientation="h",
                   title="Error Field Frequency",
                   color="count", color_continuous_scale="Reds")
    fig2.show()
else:
    print("no error data")

2. Error frequency chart


This chart shows that errors in transaction data are concentrated in the counterparty identification fields, with the most issues found in `other_counterparty_lei`, `reporting_counterparty_lei` and `uti`; margin and collateral fields follow; whilst basic fields such as timestamps and currency have relatively good data quality. This highlights weaknesses in the system’s data governance capabilities in three key areas: counterparty identity verification, transaction identification, and derivatives margin management. These areas represent priority areas for future data cleansing and system optimisation.

In [6]:
# Chart 3：Asset class breakdown
print("3. Asset class breakdown")

import plotly.express as px
import pandas as pd
import json

with open("result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

parsed_trades = data.get("parsed_trades", [])
compliance_results = data.get("compliance_results", [])


trade_asset = {}
for t in parsed_trades:
    trade_asset[t["trade_id"]] = t["asset_class"]


trade_status = {}
for item in compliance_results:
    tid = item["trade_id"]
    regime = item["regime"]
    status = item["status"]
    if tid not in trade_status:
        trade_status[tid] = {"CFTC": None, "EMIR": None}
    trade_status[tid][regime] = (status == "COMPLIANT")


rows = []
for tid in trade_status:
    asset = trade_asset[tid]
    cftc_ok = trade_status[tid]["CFTC"]
    emir_ok = trade_status[tid]["EMIR"]

    rows.append({"asset": asset, "regime": "CFTC", "ok": cftc_ok})
    rows.append({"asset": asset, "regime": "EMIR", "ok": emir_ok})

df = pd.DataFrame(rows)


grouped = df.groupby(["asset", "regime"]).agg(
    total=("ok", "count"),
    compliant=("ok", "sum")
).reset_index()

grouped["rate"] = (grouped["compliant"] / grouped["total"] * 100).round(0)

asset_total_trades = df.groupby("asset")["regime"].count() // 2
x_labels = [f"{a}<br>(n={asset_total_trades[a]})" for a in grouped["asset"].unique()]

fig = px.bar(
    grouped,
    x="asset",
    y="rate",
    color="regime",
    barmode="group",
    text="rate",
    title="Asset Class Breakdown: Compliance Rate by Regime",
    color_discrete_map={"CFTC": "#6366f1", "EMIR": "#f87171"}
)

fig.update_traces(
    texttemplate="%{y}%",
    textposition="outside",
    textfont_size=12
)

fig.update_layout(
    plot_bgcolor="#121212",
    paper_bgcolor="#121212",
    font_color="white",
    yaxis_range=[0, 105],
    xaxis=dict(ticktext=x_labels, tickvals=grouped["asset"].unique()),
    margin=dict(b=120)
)

fig.show()

3. Asset class breakdown


This chart illustrates the variation in compliance rates by asset class and regulatory jurisdiction: compliant transactions were recorded only in the Credit class (one transaction compliant with CFTC, with a compliance rate of 25% for products of the same type) and the Rates class (one transaction compliant with CFTC, with a compliance rate of 10% for products of the same type); the CFTC/EMIR compliance rate for all other asset classes was 0%. Overall, the system provides insufficient support for cross-jurisdictional compliance across most asset classes, with only traditional credit and interest rate derivatives showing limited compliance under the CFTC.

In [5]:
import pandas as pd
from IPython.display import display, HTML

print("4. Classification frontier panel")

table_data = []
for trade_id in novel_trade_ids:
    cftc_status = "N/A"
    emir_status = "N/A"
    cftc_notes = []
    emir_notes = []

    for c in compliance_results:
        if c["trade_id"] == trade_id:
            if c["regime"] == "CFTC":
                cftc_status = c["status"]
                for f in c.get("findings", []):
                    cftc_notes.append(f["message"])
            elif c["regime"] == "EMIR":
                emir_status = c["status"]
                for f in c.get("findings", []):
                    emir_notes.append(f["message"])

    table_data.append({
        "Trade ID": trade_id,
        "CFTC Status": cftc_status,
        "CFTC Compliance Notes": "\n".join(cftc_notes) if cftc_notes else "No issues",
        "EMIR Status": emir_status,
        "EMIR Compliance Notes": "\n".join(emir_notes) if emir_notes else "No issues"
    })

df_table = pd.DataFrame(table_data)

def highlight_status(val):
    if val == "PASS":
        return "background-color: #4ade80; color: white;"
    elif val == "CONDITIONAL":
        return "background-color: #fbbf24; color: #1f2937;"
    elif val == "NOT_APPLICABLE":
        return "background-color: #9ca3af; color: white;"
    else:
        return "background-color: #f87171; color: white;"

styled_table = df_table.style.map(
    highlight_status, subset=["CFTC Status", "EMIR Status"]
).hide(axis='index') \
.set_properties(**{
    'white-space': 'pre-wrap',
    'border': '1px solid #374151',
    'padding': '10px',
    'color': '#e5e7eb',
    'text-align': 'left'
}).set_table_styles([{
    'selector': 'th',
    'props': [
        ('background-color', '#1f2937'),
        ('color', '#f9fafb'),
        ('text-align', 'left'),
        ('padding', '12px')
    ]
}]).set_caption("T026-T028 Jurisdictional Asymmetry Compliance Table")

display(styled_table)

4. Classification frontier panel


Trade ID,CFTC Status,CFTC Compliance Notes,EMIR Status,EMIR Compliance Notes
T026,CONDITIONAL,"EventContract has no ANNA-DSB OTC UPI product definition; because it is traded on a CFTC-regulated DCM, CFTC treatment is CONDITIONAL pending classification.",NOT_APPLICABLE,EventContract is treated as outside EMIR OTC derivative reporting scope due to gambling classification under European national frameworks; normal EMIR field validation is not applied.
T027,NOT_APPLICABLE,"EventContract has no ANNA-DSB OTC UPI product definition; because it is not traded on a CFTC-regulated DCM, CFTC OTC reporting is NOT_APPLICABLE in this project.",NOT_APPLICABLE,EventContract is treated as outside EMIR OTC derivative reporting scope due to gambling classification under European national frameworks; normal EMIR field validation is not applied.
T028,CONDITIONAL,"EventContract has no ANNA-DSB OTC UPI product definition; because it is traded on a CFTC-regulated DCM, CFTC treatment is CONDITIONAL pending classification.",NOT_APPLICABLE,EventContract is treated as outside EMIR OTC derivative reporting scope due to gambling classification under European national frameworks; normal EMIR field validation is not applied.
